In [3]:
import math

In [4]:
def haversine_distance(origin, destination):
  lat1, lon1 = origin
  lat2, lon2 = destination
  radius = 6371000
  dlat = math.radians(lat2-lat1)
  dlon = math.radians(lon2-lon1)
  a = math.sin(dlat/2) * math.sin(dlat/2) + math.cos(math.radians(lat1)) \
    * math.cos(math.radians(lat2)) * math.sin(dlon/2) * math.sin(dlon/2)
  c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
  distance = radius * c
  return distance

In [6]:
bangalore = (12.9716, 77.5946)
mysore = (12.2958, 76.6394)

distance = haversine_distance(bangalore, mysore)
print(distance/1000)

128.016856022779


Prompt for Claude AI

```
I have a formula for distance calculation

def haversine_distance(origin, destination):
  lat1, lon1 = origin
  lat2, lon2 = destination
  radius = 6371000
  dlat = math.radians(lat2-lat1)
  dlon = math.radians(lon2-lon1)
  a = math.sin(dlat/2) * math.sin(dlat/2) + math.cos(math.radians(lat1)) \
    * math.cos(math.radians(lat2)) * math.sin(dlon/2) * math.sin(dlon/2)
  c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
  distance = radius * c
  return distance

Write a function vincenty_distance() with the same structure that implement the vincenty's formula for distance calculation on an ellipsoid
```


In [1]:
import math

def haversine_distance(origin, destination):
    lat1, lon1 = origin
    lat2, lon2 = destination
    EARTH_RADIUS_M = 6_371_000
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2
         + math.cos(math.radians(lat1))
         * math.cos(math.radians(lat2))
         * math.sin(dlon / 2) ** 2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return EARTH_RADIUS_M * c

def vincenty_distance(origin, destination, max_iterations=200, tolerance=1e-12):
    A = 6_378_137.0
    F = 1 / 298.257223563
    B = A * (1 - F)
    lat1, lon1 = map(math.radians, origin)
    lat2, lon2 = map(math.radians, destination)
    U1 = math.atan((1 - F) * math.tan(lat1))
    U2 = math.atan((1 - F) * math.tan(lat2))
    sin_U1, cos_U1 = math.sin(U1), math.cos(U1)
    sin_U2, cos_U2 = math.sin(U2), math.cos(U2)
    L = lon2 - lon1
    lam = L
    for _ in range(max_iterations):
        sin_lam, cos_lam = math.sin(lam), math.cos(lam)
        sin_sigma = math.sqrt((cos_U2 * sin_lam) ** 2 + (cos_U1 * sin_U2 - sin_U1 * cos_U2 * cos_lam) ** 2)
        if sin_sigma == 0:
            return 0.0
        cos_sigma = sin_U1 * sin_U2 + cos_U1 * cos_U2 * cos_lam
        sigma = math.atan2(sin_sigma, cos_sigma)
        sin_alpha = (cos_U1 * cos_U2 * sin_lam) / sin_sigma
        cos2_alpha = 1 - sin_alpha ** 2
        cos_2sigma_m = (cos_sigma - 2 * sin_U1 * sin_U2 / cos2_alpha) if cos2_alpha != 0 else 0.0
        C = (F / 16) * cos2_alpha * (4 + F * (4 - 3 * cos2_alpha))
        lam_prev = lam
        lam = L + (1 - C) * F * sin_alpha * (sigma + C * sin_sigma * (cos_2sigma_m + C * cos_sigma * (-1 + 2 * cos_2sigma_m ** 2)))
        if abs(lam - lam_prev) < tolerance:
            break
    else:
        return None
    u2 = cos2_alpha * (A ** 2 - B ** 2) / B ** 2
    k1 = (math.sqrt(1 + u2) - 1) / (math.sqrt(1 + u2) + 1)
    A_coeff = (1 + k1 ** 2 / 4) / (1 - k1)
    B_coeff = k1 * (1 - 3 * k1 ** 2 / 8)
    delta_sigma = B_coeff * sin_sigma * (cos_2sigma_m + (B_coeff / 4) * (cos_sigma * (-1 + 2 * cos_2sigma_m ** 2) - (B_coeff / 6) * cos_2sigma_m * (-3 + 4 * sin_sigma ** 2) * (-3 + 4 * cos_2sigma_m ** 2)))
    return B * A_coeff * (sigma - delta_sigma)


# ── Test ──────────────────────────────────────────────────────────────────────
san_francisco = (37.7749, -122.4194)
new_york      = (40.661,   -73.944)

hav  = haversine_distance(san_francisco, new_york)
vinc = vincenty_distance(san_francisco, new_york)
diff = abs(hav - vinc)

print(f"Haversine  : {hav/1000:>10.4f} km")
print(f"Vincenty   : {vinc/1000:>10.4f} km")
print(f"Difference : {diff:>10.4f} m  ({diff/vinc*100:.4f}%)")

Haversine  :  4135.3746 km
Vincenty   :  4145.4470 km
Difference : 10072.3604 m  (0.2430%)
